In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error

In [8]:
train = pd.read_csv(r"C:\Users\hp\Desktop\Data_analysis-and-ML\train (2).csv")
test = pd.read_csv(r"C:\Users\hp\Desktop\Data_analysis-and-ML\test (2).csv")
transcripts = pd.read_csv(r"C:\Users\hp\Desktop\Data_analysis-and-ML\transcripts.csv")

In [9]:
train = train[train["views"] >= 0].copy()

text_cols = [
    "title",
    "description",
    "main_speaker",
    "speaker_occupation",
    "event",
    "tags",
    "ratings"
]

for col in text_cols:
    if col not in train.columns:
        train[col] = ""
    if col not in test.columns:
        test[col] = ""


train[text_cols] = train[text_cols].fillna("")   
test[text_cols] = test[text_cols].fillna("")

train["all_text"] = train[text_cols].agg(" ".join, axis=1)
test["all_text"] = test[text_cols].agg(" ".join, axis=1)

In [10]:
num_cols = ["duration", "languages", "num_speaker"]

for col in num_cols:
    if col not in train.columns:
        train[col] = 0
    if col not in test.columns:
        test[col] = 0

train_medians = train[num_cols].median()

train[num_cols] = train[num_cols].fillna(train_medians)
test[num_cols] = test[num_cols].fillna(train_medians)

In [11]:
y = np.log1p(train["views"])

In [12]:
X_train, X_valid, y_train, y_valid = train_test_split(
    train,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=20000, ngram_range=(1, 2)), "all_text"),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=3.0))
])
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [14]:
model.fit(train, y)
test_pred_log = model.predict(test)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 0, None)
print(test.columns)

Index(['Unnamed: 0', 'description', 'duration', 'event', 'film_date',
       'languages', 'main_speaker', 'name', 'num_speaker', 'published_date',
       'ratings', 'related_talks', 'speaker_occupation', 'tags', 'title',
       'url', 'id', 'all_text'],
      dtype='object')


In [15]:
id_col = "id"   # change if needed

submission = pd.DataFrame({
    id_col: test[id_col],
    "views": test_pred
})

submission.to_csv("submission.csv", index=False)
print(submission.head())

     id         views
0    56  8.091711e+05
1   194  6.362415e+05
2  2225  1.217774e+06
3   233  4.521712e+05
4  1902  2.122077e+06
